## Загрузка очищенных данных

На этом этапе мы загружаем предварительно очищенный датасет с помощью функции `load_cleaned_data()` из модуля `src.data_loader`. 

- Импортируются необходимые библиотеки (pandas, numpy)
- Добавляется путь к модулям `src`
- Загружаются данные, которые уже прошли первичную обработку
- Проверяется размерность датасета

Данные содержат информацию о химических соединениях, их дескрипторах и биологической активности (IC50, CC50, SI).

In [ ]:

import sys
import os
import pandas as pd
import numpy as np

# Добавляем корень проекта в путь
sys.path.append(os.path.dirname(os.getcwd()))

from src import load_cleaned_data
from src import prepare_data_for_modeling

# 1. Загрузка данных
df = load_cleaned_data()
df.shape

(998, 171)

## Создание целевых переменных для классификации

Исходные данные содержат непрерывные значения активности, но для некоторых задач нам нужны бинарные метки. 

**Регрессионные таргеты (непрерывные значения):**
- `pIC50` - отрицательный логарифм IC50 (ингибирующая концентрация)
- `pCC50` - отрицательный логарифм CC50 (цитотоксическая концентрация)  
- `log_SI` - логарифм селективности (SI = CC50/IC50)

**Классификационные таргеты (бинарные метки):**
1. `IC50_binary` - активность выше/ниже медианы
2. `CC50_binary` - цитотоксичность выше/ниже медианы
3. `SI_binary` - селективность выше/ниже медианы
4. `SI_8_binary` - селективность > 8 (пороговое значение)


In [ ]:
REGRESSION_TARGETS = ['pIC50', 'pCC50', 'log_SI']
CLASSIFICATION_TARGETS = ['IC50_binary', 'CC50_binary', 'SI_binary', 'SI_8_binary']

# Создаем бинарные метки для классификации
median_IC50 = df['IC50, mM'].median()
median_CC50 = df['CC50, mM'].median()
median_SI = df['SI'].median()

df['IC50_binary'] = (df['IC50, mM'] > median_IC50).astype(int)
df['CC50_binary'] = (df['CC50, mM'] > median_CC50).astype(int)
df['SI_binary'] = (df['SI'] > median_SI).astype(int)
df['SI_8_binary'] = (df['SI'] > 8).astype(int)

df.shape

(998, 175)

## Подготовка данных для моделирования

Этот блок выполняет финальную подготовку данных перед обучением моделей:

**Шаги обработки (внутри функции `prepare_data_for_modeling`):**
1. **Разделение на признаки и таргеты** - отделение дескрипторов от целевых переменных
2. **Разбиение на train/test** - 80% тренировочных, 20% тестовых данных (random_state=42 для воспроизводимости)
3. **Масштабирование признаков** - стандартизация с помощью StandardScaler (только на train, затем применяется к test)
4. **Сохранение подготовленных данных** в папку `data/processed/`

**Сохраняемые объекты:**
- `X_train_scaled.pkl` - тренировочные признаки (масштабированные)
- `X_test_scaled.pkl` - тестовые признаки (масштабированные)
- `y_train_reg.pkl` - регрессионные таргеты для обучения
- `y_test_reg.pkl` - регрессионные таргеты для теста
- `y_train_clf.pkl` - классификационные таргеты для обучения
- `y_test_clf.pkl` - классификационные таргеты для теста
- `scaler.pkl` - объект StandardScaler для использования в будущем
- `feature_names.pkl` - список названий признаков

После сохранения данные готовы к использованию в моделях машинного обучения.

In [ ]:
# Подготовка для моделирования
data = prepare_data_for_modeling(
    df,
    reg_targets=REGRESSION_TARGETS,
    clf_targets=CLASSIFICATION_TARGETS,
    test_size=0.2,
    random_state=42
)

# Сохраняем подготовленные данные на диск
import joblib
SAVE_PATH = '../data/processed/'

joblib.dump(data['X_train'], f'{SAVE_PATH}X_train_scaled.pkl')
joblib.dump(data['X_test'], f'{SAVE_PATH}X_test_scaled.pkl')
joblib.dump(data['y_train_reg'], f'{SAVE_PATH}y_train_reg.pkl')
joblib.dump(data['y_test_reg'], f'{SAVE_PATH}y_test_reg.pkl')
joblib.dump(data['y_train_clf'], f'{SAVE_PATH}y_train_clf.pkl')
joblib.dump(data['y_test_clf'], f'{SAVE_PATH}y_test_clf.pkl')
joblib.dump(data['scaler'], f'{SAVE_PATH}scaler.pkl')
joblib.dump(data['feature_names'], f'{SAVE_PATH}feature_names.pkl')

['../data/processed/feature_names.pkl']